In [1]:
import torch
import torch.nn as nn
import math
import pretty_midi
import numpy as np
import os

MIDI Data Reading:

In [2]:
#method takes a midi path, and returns a list of pitches, and a list of intervals above those pitches
def midiToArrays(path):
    CPT1 = pretty_midi.PrettyMIDI(path)
    
    cantus = CPT1.instruments[1]
    discantus = CPT1.instruments[0]

    cantusNotes = sorted(cantus.notes, key=lambda n: n.start)
    discantusNotes = sorted(discantus.notes, key=lambda n: n.start)

    cantusPitches = [note.pitch for note in cantusNotes]
    discantusPitches = [note.pitch for note in discantusNotes]
    discantusIntervals = [d - c for c, d in zip(cantusPitches, discantusPitches)]

    return cantusPitches, discantusIntervals

cant, dis = midiToArrays("data/major/CPT1.mid")
print(dis)
print(cant)


[24, 16, 20, 12, 16, 21, 24]
[48, 55, 52, 55, 53, 50, 48]


C:\Users\Wreck\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


In [3]:
import os
import torch
from torch.nn.utils.rnn import pad_sequence

basePath = "data"

cantusArray = []
intervalArray = []

for root, dirs, files in os.walk(basePath):
    for filename in files:
        currPath = os.path.join(root, filename)
        cantus, intervals = midiToArrays(currPath)

        cantusArray.append(torch.tensor(cantus, dtype=torch.long))
        intervalArray.append(torch.tensor(intervals, dtype=torch.long))

#pad with 0's so everything is same length
cantusTensor = pad_sequence(cantusArray, batch_first=True, padding_value=0)
intervalsTensor = pad_sequence(intervalArray, batch_first=True, padding_value=0)

print(cantusTensor)


tensor([[48, 55, 51,  ...,  0,  0,  0],
        [47, 49, 56,  ..., 50, 49, 47],
        [50, 47, 45,  ...,  0,  0,  0],
        ...,
        [52, 48, 43,  ..., 52,  0,  0],
        [48, 49, 51,  ..., 48,  0,  0],
        [47, 54, 48,  ...,  0,  0,  0]])


Data Augmentation:

Transpose all canti up and down to multiply our dataset. We check to make sure it doesn't violate any range rules and only include the ones that work.

In [4]:
MINPITCH = 36 #C2
MAXPITCH = 84 #C6


def transpose(cantus, intervals, k):
    mask = cantus != 0 #don't transpose the 0

    tempCantus = cantus.clone()
    tempIntervals = intervals.clone()

    tempCantus[mask] += k

    #ensure new transposition is not too high or low
    if tempCantus[mask].min() < MINPITCH:
        return None
    if tempCantus[mask].max() > MAXPITCH:
        return None

    return tempCantus, tempIntervals


In [5]:
newCantusList = []
newIntervalList = []

for cantus, intervals in zip(cantusTensor, intervalsTensor):
    for k in range(-4, 5):
        result = transpose(cantus, intervals, k)
        if result is None:
            continue
        newCantus, newIntervals = result
        newCantusList.append(newCantus)
        newIntervalList.append(newIntervals)
    
augCantusTensor = torch.stack(newCantusList)
augIntervalTensor = torch.stack(newIntervalList)

print(len(augCantusTensor))
numIntervals = int(augIntervalTensor.max()) + 1  #+1 to include 0 padding
print("number of intervals", numIntervals)

augCantusTensor = augCantusTensor.transpose(0, 1)
augIntervalTensor = augIntervalTensor.transpose(0, 1)


1347
number of intervals 34


Transformer Architecture:

In [6]:
import torch
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self, numPitches, numIntervals =34 , embedSize = 64, maxLen = 128, numHeads = 4, layers = 2, dimFeedforward = 128):
        super().__init__()

        #embeddings for pitches, intervals, and positions
        self.cantusEmbed = nn.Embedding(numPitches, embedSize, padding_idx=0)
        self.intervalEmbed = nn.Embedding(numIntervals, embedSize, padding_idx=0)
        self.posEmbed = nn.Embedding(maxLen, embedSize)

        #decoder predicts the next interval
        decodeLayer = nn.TransformerDecoderLayer(
            d_model=embedSize, nhead=numHeads, dim_feedforward=dimFeedforward
        )

        self.decoder=nn.TransformerDecoder(decodeLayer,num_layers=layers)
        
        #outputs logits of each interval
        self.output = nn.Linear(embedSize, numIntervals)

    def forward(self, cantusSeq, intervalSeq):
        cantusEmbed = self.cantusEmbed(cantusSeq)
        intervalEmbed = self.intervalEmbed(intervalSeq)

        seqLenTgt, batch = intervalSeq.shape
        seqLenMem, _ = cantusSeq.shape
        
        #positional embeddings
        positionsTgt = torch.arange(seqLenTgt, device=cantusSeq.device).unsqueeze(1)
        intervalEmbed = intervalEmbed + self.posEmbed(positionsTgt)

        positionsMem = torch.arange(seqLenMem, device=cantusSeq.device).unsqueeze(1)
        cantusEmbed = cantusEmbed + self.posEmbed(positionsMem)

        #causal mask for target only
        mask = nn.Transformer.generate_square_subsequent_mask(seqLenTgt).to(cantusSeq.device)

        cantusPaddingMask = (cantusSeq == 0).transpose(0, 1)
        intervalPaddingMask = (intervalSeq == 0).transpose(0, 1)

        decoderOut = self.decoder(
            tgt=intervalEmbed,
            memory=cantusEmbed,
            tgt_mask=mask,
            tgt_key_padding_mask=intervalPaddingMask,
            memory_key_padding_mask=cantusPaddingMask
        )

        logits = self.output(decoderOut)
        return logits


In [7]:
seqLength = 16
batch = 2
numPitches = 128
numIntervals = 34

cantusSeq = torch.randint(0, numPitches, (seqLength, batch))
intervalSeq = torch.randint(0, numIntervals, (seqLength, batch))

model = Transformer(numPitches, numIntervals)
logits = model(cantusSeq, intervalSeq)

print(logits.shape)


C:\Users\Wreck\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\nn\functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


torch.Size([16, 2, 34])


C:\Users\Wreck\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\nn\functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Splitting the data randomly 80-20 for training and val

In [8]:
#random data split
numExamples = augCantusTensor.shape[1]
split = int(0.8 * numExamples)

perm = torch.randperm(numExamples)

augCantusTensor   = augCantusTensor[:, perm]
augIntervalTensor = augIntervalTensor[:, perm]

trainCantus = augCantusTensor[:, :split]
trainIntervals = augIntervalTensor[:, :split]

valCantus = augCantusTensor[:, split:]
valIntervals = augIntervalTensor[:, split:]


Define optimizer and loss metric, training loop:

In [104]:
model = Transformer(numPitches, numIntervals)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

startToken = 1
inputIntervals = torch.cat([
    torch.full((1, augIntervalTensor.shape[1]), startToken, dtype=torch.long),
    augIntervalTensor[:-1, :]
], dim=0)


epochs = 3000
bestVal = float("inf")
patience = 100
patienceCount = 0

for epoch in range(epochs):

    #training loss
    model.train()
    optimizer.zero_grad()

    trainInput = torch.cat([
        torch.full((1, trainIntervals.shape[1]), startToken, dtype=torch.long),
        trainIntervals[:-1, :]
    ], dim=0)

    trainLogits = model(trainCantus, trainInput)

    trainLoss = criterion(
        trainLogits.reshape(-1, numIntervals),
        trainIntervals.reshape(-1)
    )

    trainLoss.backward()
    optimizer.step()

    #validation loss
    model.eval()
    with torch.no_grad():
        valInput = torch.cat([
            torch.full((1, valIntervals.shape[1]), startToken, dtype=torch.long),
            valIntervals[:-1, :]
        ], dim=0)

        valLogits = model(valCantus, valInput)

        valLoss = criterion(
            valLogits.reshape(-1, numIntervals),
            valIntervals.reshape(-1)
        )

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {trainLoss.item():.4f} | "
        f"Val Loss: {valLoss.item():.4f}"
    )

    #stop if val loss has not progressed 50 times
    if valLoss < bestVal:
        bestVal = valLoss
        patienceCount = 0
        torch.save(model.state_dict(), "best.pt")
    else:
        patienceCount += 1
        if patienceCount >= patience:
            print("Early stop")
            break


Epoch 1 | Train Loss: 3.7167 | Val Loss: 3.4468
Epoch 2 | Train Loss: 3.4752 | Val Loss: 3.2367
Epoch 3 | Train Loss: 3.2719 | Val Loss: 3.0631
Epoch 4 | Train Loss: 3.1008 | Val Loss: 2.9221
Epoch 5 | Train Loss: 2.9590 | Val Loss: 2.8078
Epoch 6 | Train Loss: 2.8426 | Val Loss: 2.7141
Epoch 7 | Train Loss: 2.7487 | Val Loss: 2.6359
Epoch 8 | Train Loss: 2.6726 | Val Loss: 2.5694
Epoch 9 | Train Loss: 2.6021 | Val Loss: 2.5118
Epoch 10 | Train Loss: 2.5411 | Val Loss: 2.4608
Epoch 11 | Train Loss: 2.4878 | Val Loss: 2.4150
Epoch 12 | Train Loss: 2.4423 | Val Loss: 2.3732
Epoch 13 | Train Loss: 2.3970 | Val Loss: 2.3339
Epoch 14 | Train Loss: 2.3592 | Val Loss: 2.2964
Epoch 15 | Train Loss: 2.3180 | Val Loss: 2.2603
Epoch 16 | Train Loss: 2.2817 | Val Loss: 2.2258
Epoch 17 | Train Loss: 2.2457 | Val Loss: 2.1934
Epoch 18 | Train Loss: 2.2109 | Val Loss: 2.1633
Epoch 19 | Train Loss: 2.1796 | Val Loss: 2.1359
Epoch 20 | Train Loss: 2.1492 | Val Loss: 2.1110
Epoch 21 | Train Loss: 2.1230

Generate a single example using the model:

In [9]:
genModel = Transformer(numPitches, numIntervals)
distinct = torch.load("best.pt")
genModel.load_state_dict(distinct)
genModel.eval()

#get one cantus from the data
cantus = augCantusTensor[:, 4]
cantus = cantus.unsqueeze(1)
#print(cantus)

START = 1
generatedIntervals = torch.tensor([[START]], dtype=torch.long)
realLen = (cantus.squeeze(1) != 0).sum().item()

for t in range(realLen):
    with torch.no_grad():
        #pass the full timestep each time
        logits = genModel(cantus, generatedIntervals)
        nextLogits = logits[-1, 0]
        nextInterval = torch.argmax(nextLogits).item()
        
        generatedIntervals = torch.cat([generatedIntervals, torch.tensor([[nextInterval]], dtype=torch.long)], dim=0)

#remove the start token
generatedIntervals = generatedIntervals[1:, 0]

#compute the discantus with the generated intervals
discantus = cantus.squeeze(1)[:realLen] + generatedIntervals

#convert for readability
noteNames = ['C','C#','D','D#', 'E','F','F#','G','G#','A','A#','B']

def midi_to_note(midiNum):
    octave = (midiNum // 12) - 1
    note = noteNames[midiNum % 12]
    return f"{note}{octave}"

print("Cantus  | Discantus")

for c, d in zip(cantus.squeeze(1)[:realLen], discantus):
    cName = midi_to_note(c.item())
    dName = midi_to_note(d.item())
    print(f"{cName:>7} | {dName:>9}")

Cantus  | Discantus
    C#3 |       C#4
     C3 |       D#4
    C#3 |        F4
    G#3 |        C4
    F#3 |       A#3
    D#3 |        C4
    C#3 |       C#4


Write generated example to MIDI File:

In [10]:
from mido import Message, MidiFile, MidiTrack, MetaMessage

QUARTER_NOTE = 480
WHOLE_NOTE = 480 * 4
VELOCITY = 127

file = MidiFile(ticks_per_beat = QUARTER_NOTE)

discantusTrack = MidiTrack()
cantusTrack = MidiTrack()


file.tracks.append(discantusTrack)
file.tracks.append(cantusTrack)

for cantusNote, disCantusNote in zip(cantus.squeeze(1)[:realLen],discantus):
    c = int(cantusNote.item())
    d = int(disCantusNote)

    cantusTrack.append(Message('note_on', note=c, velocity = VELOCITY, time=0))
    cantusTrack.append(Message('note_off', note=c, velocity = VELOCITY, time=WHOLE_NOTE))

    discantusTrack.append(Message('note_on', note=d, velocity = VELOCITY, time=0))
    discantusTrack.append(Message('note_off', note=d, velocity = VELOCITY, time=WHOLE_NOTE))


file.save("CounterTest.mid")
